In [18]:
from nazi_symbols_classification.training.data_preparation import (
    download_data_from_roboflow, reorganise_images, remap_images, get_image_paths
)

In [2]:
class_map = {
    # "non-nazi": None,
    "neo_nazi": ['national_rebirth_poland', 'combat_18_emblem', 'atomwaffen',
                 'kolovrat', 'volksfront_emblem', 'celtic_cross', 'hammerskins',
                 'identitaere_bewegung_emblem', 'blood_honor_emblem', 'golden_dawn'],
    "siegrune": ['doppelsiegrune', 'siegrune',]
}

In [3]:
remap_images(class_map, "./datasets/nazi-symbols-classification", sub_folders=("train", "test", "valid"))

In [7]:
!cp -r ./datasets/nazi-symbols-classification/valid/judenstern ./datasets/nazi-symbols-classification/test/judenstern
!cp -r ./datasets/nazi-symbols-classification/valid/sturmabteilung_emblem ./datasets/nazi-symbols-classification/test/sturmabteilung_emblem

In [19]:
image_paths = get_image_paths(path="./datasets/nazi-symbols-classification",
                              sub_folders=("train", "test", "valid"))
len(image_paths)

3370

In [5]:
len(image_paths)

3370

In [20]:
import requests 

def classify_document(doc_path, prompts, open_clip_api_endpoint):

    with open(doc_path, "rb") as doc_file:
        files = dict(doc_file=(doc_path, doc_file, "multipart/form-data"), )
        response = requests.post(open_clip_api_endpoint,
                                 files=files,
                                 data=dict(prompts=prompts),
                                 timeout=10)
        response.raise_for_status()
    result = response.json()
    result_dict = dict(zip(result["prompts"], result["probs"]))
    return result_dict

In [21]:
prompts = {
    "A black sun symbol, consisting of concentric circles with radiating, rune-like spokes, associated with Nazi occultism.": "black_sun",
    "The British Union of Fascists logo, a black lightning bolt set within a white circle on a dark background, symbolizing their fascist ideology.": "british_union_of_fascist",
    "A broken sun cross symbol, featuring a circle divided into four or more segments by straight lines, often associated with white supremacist or neo-Nazi groups.": "broken_sun_cross",
    "Historical images of Adolf Hitler addressing crowds, giving speeches, or leading Nazi rallies during the 1930s and 1940s.": "hitler",
    "Images of individuals performing the Hitler salute during historical Nazi Germany events, characterized by a raised right arm held at an angle.": "hitler_salute",
    "Images of the Judenstern, the yellow Star of David badge used during the Holocaust, often featuring the word 'Jude' in black lettering in the center.": "judenstern",
    "Imagery of neo-Nazi groups featuring hate symbols like swastikas, Black Sun, Siegrune, or Celtic Cross on flags, banners, clothing, or graffiti, often seen at rallies, protests, or in propaganda materials promoting white supremacy and far-right ideology.": "kolovrat",
    "A single angular rune shaped like a lightning bolt or elongated 'S,' used in Nazi and neo-Nazi iconography.": "siegrune",
    "A skull and crossbones insignia, often used by the Nazi SS, with a sinister and militaristic design.": "ss_skull",
    "an image with the sign of sturmabteilung emblem": "sturmabteilung_emblem",
    "A black swastika symbol with arms bent at 90 degrees, typically rotated at a 45-degree angle, often shown on a red circular background or a white circle, used during World War II by Nazi Germany.": "swastika",
    "The Wolfsangel symbol, resembling a hook-like rune, used by Nazi groups and German military units during World War II.": "wolfsangel",
    "an image containing no nazi related content": "non-nazi",
}

In [8]:
classify_document(image_paths[0], prompts.keys(), "http://localhost:8080/api/v1/classify")

{'A black sun symbol, consisting of concentric circles with radiating, rune-like spokes, associated with Nazi occultism.': 7.661258132429793e-05,
 'The British Union of Fascists logo, a black lightning bolt set within a white circle on a dark background, symbolizing their fascist ideology.': 7.15955684427172e-05,
 'A broken sun cross symbol, featuring a circle divided into four or more segments by straight lines, often associated with white supremacist or neo-Nazi groups.': 0.890302836894989,
 'Historical images of Adolf Hitler addressing crowds, giving speeches, or leading Nazi rallies during the 1930s and 1940s.': 2.5318098312299142e-11,
 'Images of individuals performing the Hitler salute during historical Nazi Germany events, characterized by a raised right arm held at an angle.': 3.786764235513829e-09,
 "Images of the Judenstern, the yellow Star of David badge used during the Holocaust, often featuring the word 'Jude' in black lettering in the center.": 0.00013370049418881536,
 'I

In [27]:
for image_path in image_paths:
    if "british_union_of_fascist" not in image_path:
        continue
    result = classify_document(image_path, prompts.keys(), "http://localhost:8080/api/v1/classify")
    print("-----------------------")
    print(image_path)
    for k, v in result.items():
        print(prompts[k], " : ", v)
    print("-----------------------")


-----------------------
./datasets/nazi-symbols-classification/train/british_union_of_fascist/13739a035233eefaaab05996246e596390403fa2_full_jpg.rf.b4576f6f74a6fb3e0de4ab319632b8bb.jpg
black_sun  :  7.275642701642937e-07
british_union_of_fascist  :  0.313032865524292
broken_sun_cross  :  5.428741860669106e-05
hitler  :  0.006858156528323889
hitler_salute  :  0.23782646656036377
judenstern  :  2.8938296736669145e-07
kolovrat  :  0.22430069744586945
siegrune  :  0.0003450285876169801
ss_skull  :  1.4953220670577139e-05
sturmabteilung_emblem  :  0.00732032535597682
swastika  :  0.0004986506537534297
wolfsangel  :  0.04747610166668892
non-nazi  :  0.1622714251279831
-----------------------
-----------------------
./datasets/nazi-symbols-classification/train/british_union_of_fascist/8e0fcf32d7023ef409b98f1bf9ed0e61a1409fdb_full_jpg.rf.afc76861911fba24e201c4e12b072171.jpg
black_sun  :  1.1980023195690137e-08
british_union_of_fascist  :  0.9989259839057922
broken_sun_cross  :  5.39545471838209

In [9]:
image_paths[0]

'./datasets/nazi-symbols-classification/train/neo_nazi/80a4f72fbb195ffcf75ca3083e65e42fbf42e8aa_full_jpg.rf.58b464c6fb2414a60aeb9c7dc155686e.jpg'

In [10]:
result = dict()

for image_path in image_paths:
    classify_result = classify_document(image_path, prompts.keys(), "http://localhost:8080/api/v1/classify")
    result[image_path] = classify_result

In [11]:
import json


with open("open_clip_result_v2.json", "w") as f:
    json.dump(result, f)

In [12]:
import json


with open("open_clip_result_v2.json", "r") as f:
    result = json.load(f)

In [13]:
import os

classification_dict = dict()
for image_path, classify_result in result.items():
    top5 = sorted(classify_result.items(), key=lambda x: x[1], reverse=True)[:5]
    predicted = [prompts[prompt] for prompt, prob in top5 if prob >= 0.01]
    if not predicted:
        predicted = [prompts[prompt] for prompt, prob in top5[:1]]
    label = os.path.basename(os.path.dirname(image_path))
    classification_dict[image_path] = dict(predicted=predicted, label=label)

classification_dict

{'./datasets/nazi-symbols-classification/train/neo_nazi/80a4f72fbb195ffcf75ca3083e65e42fbf42e8aa_full_jpg.rf.58b464c6fb2414a60aeb9c7dc155686e.jpg': {'predicted': ['broken_sun_cross',
   'wolfsangel'],
  'label': 'neo_nazi'},
 './datasets/nazi-symbols-classification/train/neo_nazi/973px-C18_Blood_and_Honour_triskele_any_C20_B-H-correlation_is_disputed_png_jpg.rf.f354d1976ef1827bc931d0ddfdfce392.jpg': {'predicted': ['ss_skull',
   'kolovrat',
   'non-nazi'],
  'label': 'neo_nazi'},
 './datasets/nazi-symbols-classification/train/neo_nazi/d46a6098e3a17ebfe0382feae32e02546436f9b9_full_jpg.rf.e1848132b964bca467fa5577e1af84e3.jpg': {'predicted': ['kolovrat',
   'broken_sun_cross',
   'swastika',
   'wolfsangel'],
  'label': 'neo_nazi'},
 './datasets/nazi-symbols-classification/train/neo_nazi/0a705f46c9d0d5a5c2610e8ef297d18e6587d2c4_full_jpg.rf.8d6c3ee6c8695af6f735a27d022c00bb.jpg': {'predicted': ['ss_skull',
   'kolovrat',
   'non-nazi'],
  'label': 'neo_nazi'},
 './datasets/nazi-symbols-clas

In [14]:
predictions_top5 = []
predictions_top1= []
labels = []

for image_path, cls_result in classification_dict.items():
    labels.append(cls_result['label'])
    predictions_top5.append(cls_result['label'] if cls_result['label'] in cls_result['predicted'] else cls_result['predicted'][0])
    predictions_top1.append(cls_result['predicted'][0])

In [15]:
from sklearn.metrics import classification_report


In [16]:
report_top1 = classification_report(labels, predictions_top1)
print(report_top1)

                          precision    recall  f1-score   support

               black_sun       0.67      0.14      0.24       152
british_union_of_fascist       0.15      0.54      0.23        13
        broken_sun_cross       0.04      0.29      0.08        21
          happy_merchant       0.00      0.00      0.00        39
                  hitler       0.67      0.37      0.48       228
           hitler_salute       0.00      0.00      0.00         6
              judenstern       0.18      1.00      0.31         4
                kolovrat       0.00      0.00      0.00         0
                neo_nazi       0.00      0.00      0.00       184
                non-nazi       0.36      0.40      0.38      1171
                siegrune       0.21      0.14      0.17       176
                ss_skull       0.50      0.61      0.55       257
   sturmabteilung_emblem       0.01      0.18      0.02        11
                swastika       0.76      0.13      0.22      1070
         

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.


In [17]:

report_top5 = classification_report(labels, predictions_top5)
print(report_top5)

                          precision    recall  f1-score   support

               black_sun       0.93      0.47      0.62       152
british_union_of_fascist       0.30      0.69      0.42        13
        broken_sun_cross       0.14      0.52      0.23        21
          happy_merchant       0.00      0.00      0.00        39
                  hitler       0.83      0.87      0.85       228
           hitler_salute       0.03      0.33      0.06         6
              judenstern       0.40      1.00      0.57         4
                kolovrat       0.00      0.00      0.00         0
                neo_nazi       0.00      0.00      0.00       184
                non-nazi       0.62      0.80      0.70      1171
                siegrune       0.46      0.30      0.36       176
                ss_skull       0.62      0.75      0.68       257
   sturmabteilung_emblem       0.05      0.27      0.08        11
                swastika       0.95      0.34      0.50      1070
         

Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
